## Étape 2 — Nettoyage

In [23]:
import json
import re
from collections import Counter

with open("../data/pages_brutes.json", encoding="utf-8") as f:
    pages = json.load(f)
print(len(pages), "pages chargées")

23 pages chargées


In [ ]:
def nettoyer(texte: str) -> str:
    texte = texte.replace("\xa0", " ")           
    texte = re.sub(r"-\n(?=[a-zàâçéèêëîïôûùü])", "", texte)   
    texte = re.sub(r"[ \t]+", " ", texte)        
    texte = re.sub(r" *\n *", "\n", texte)       
    texte = re.sub(r"\n{3,}", "\n\n", texte)     
    return texte.strip()

## Code : Q3, nettoyer toutes les pages et supprimer les pages vides

In [ ]:
pages_propres = []
for p in pages:
    texte = nettoyer(p["texte"])
    if texte:                                  
        pages_propres.append({**p, "texte": texte})

print(f"Avant : {len(pages)} pages | Après : {len(pages_propres)} pages")
print(f"Pages supprimées : {len(pages) - len(pages_propres)}")
print(f"Caractères : {sum(len(p['texte']) for p in pages)} -> {sum(len(p['texte']) for p in pages_propres)}")

In [46]:
i = 1

avant = pages[i]
apres = nettoyer(avant["texte"])

print(f"{avant['source']} (page {avant['page']})")
print("=" * 30, "AVANT", "=" * 30)
print(repr(avant["texte"][:600]))   
print("=" * 30, "APRÈS", "=" * 30)
print(repr(apres[:600]))

livret_accueil.pdf (page 1)
============================== AVANT ==============================
"Altéa Services — Document interne — Diffusion restreinte\n1\nDocument fictif réalisé à des fins pédagogiques\nLivret d'accueil du nouveau salarié\nGEN-002 — édition 2026\nBienvenue chez Altéa Services\nAltéa Services est une entreprise fictive de services aux entreprises, fondée en 2004, qui\ncompte environ 1 400 salariés sur trois sites : Paris (siège), Lyon et Nantes. Ce livret\nrassemble ce qu'il faut savoir pour bien démarrer.\nVotre premier jour\nVous êtes accueilli à 9 h 30 par votre manager et un membre du service RH. Vous recevez\nvotre badge, votre ordinateur portable et vos identifiants. Votre "
============================== APRÈS ==============================
"Altéa Services — Document interne — Diffusion restreinte\n1\nDocument fictif réalisé à des fins pédagogiques\nLivret d'accueil du nouveau salarié\nGEN-002 — édition 2026\nBienvenue chez Altéa Services\nAltéa Services est 

In [40]:
repetees = Counter()
for p in pages_propres:
    for ligne in set(p["texte"].split("\n")):
        repetees[(p["source"], ligne.strip())] += 1

for (source, ligne), n in repetees.most_common(15):
    if n > 1 and ligne:
        print(f"{n}x | {source:25} | {ligne[:80]}")

2x | livret_accueil.pdf        | Altéa Services — Document interne — Diffusion restreinte
2x | livret_accueil.pdf        | Document fictif réalisé à des fins pédagogiques
2x | plan_acces.pdf            | Altéa Services — Document interne — Diffusion restreinte
2x | plan_acces.pdf            | Document fictif réalisé à des fins pédagogiques
2x | reglement_interieur.pdf   | Altéa Services — Document interne — Diffusion restreinte
2x | reglement_interieur.pdf   | Document fictif réalisé à des fins pédagogiques
2x | charte_informatique.pdf   | Altéa Services — Document interne — Diffusion restreinte
2x | charte_informatique.pdf   | Document fictif réalisé à des fins pédagogiques
2x | conges_absences.pdf       | Altéa Services — Document interne — Diffusion restreinte
2x | conges_absences.pdf       | Document fictif réalisé à des fins pédagogiques
2x | mutuelle.pdf              | Altéa Services — Document interne — Diffusion restreinte
2x | mutuelle.pdf              | Document fictif réalis

In [41]:
with open("../data/pages_propres.json", "w", encoding="utf-8") as f:
    json.dump(pages_propres, f, ensure_ascii=False, indent=2)
print("Sauvegardé :", len(pages_propres), "pages")

Sauvegardé : 23 pages


## Q4 — Avant / après nettoyage

Page : livret_accueil.pdf, page 1. Le nettoyage (espaces, lignes vides, césures)
ne change rien sur cet extrait, le PDF étant déjà proprement extrait.

**Défaut qui reste :** l'en-tête "Altéa Services — Document interne — Diffusion
restreinte", le numéro de page isolé ("1") et le pied "Document fictif réalisé à
des fins pédagogiques" sont conservés dans le texte. Répétés sur chaque page, ils
se retrouveraient dans tous les chunks et pollueraient la recherche.

**Règle proposée :** supprimer toute ligne présente sur au moins la moitié des
pages d'un même document, et tout numéro de page isolé en début ou fin de page.

In [ ]:
from collections import Counter, defaultdict

nb_pages = Counter(p["source"] for p in pages)
lignes = defaultdict(Counter)
for p in pages:
    for l in {x.strip() for x in p["texte"].split("\n") if x.strip()}:
        lignes[p["source"]][l] += 1

source = "livret_accueil.pdf"
print(source, ":", nb_pages[source], "pages")
for l, n in lignes[source].most_common(6):
    print(f"{n}x | {l[:80]}")

In [ ]:
def retirer_repetitions(pages: list[dict]) -> list[dict]:
    """
    Retire les lignes présentes sur au moins la moitié des pages d'un document
    (en-têtes/pieds de page) et les numéros de page isolés en début/fin de page.
    """
    nb = Counter(p["source"] for p in pages)
    freq = defaultdict(Counter)
    for p in pages:
        for l in {x.strip() for x in p["texte"].split("\n") if x.strip()}:
            freq[p["source"]][l] += 1

    resultat = []
    for p in pages:
        lignes_page = p["texte"].split("\n")
        gardees = []
        for k, l in enumerate(lignes_page):
            l_nette = l.strip()
            repetee = nb[p["source"]] >= 2 and freq[p["source"]][l_nette] >= max(2, nb[p["source"]] / 2)
            numero = re.fullmatch(r"\d{1,3}", l_nette) and (k < 3 or k >= len(lignes_page) - 3)
            if not repetee and not numero:
                gardees.append(l)
        resultat.append({**p, "texte": "\n".join(gardees).strip()})
    return resultat

In [45]:
pages_nettoyees = [{**p, "texte": nettoyer(p["texte"])} for p in pages]
pages_nettoyees = retirer_repetitions(pages_nettoyees)
pages_propres = [p for p in pages_nettoyees if p["texte"]]